# Week 4: From Python Cells to an End-to-End Training Pipeline

**AIM 5012 | Effective Coding with AI**

We will not try to cover all of Python syntax today. Instead, we will use one small project to understand how Python, data, and a machine learning model connect.

```text
Data -> Inspect -> Baseline -> Train/Test Split -> Train Model -> Predict -> Evaluate -> Application Workflow
```

Project goal: classify one campus helpdesk ticket into one of four labels:

- `account`
- `billing`
- `technical`
- `other`

By the end of class, we will have a working `predict_ticket(text)` function and a small application workflow that stores predictions.


## How to Use This Notebook

This notebook is designed for students with no Python experience or only a small amount of Python experience.

- Every code cell should run from top to bottom.
- If a cell raises an error, read the error message first, then check whether earlier cells have been run.
- Before submitting or presenting, always run: **Restart Kernel and Run All**.

Required Python packages:

```text
pandas
scikit-learn
```

Optional plotting package:

```text
matplotlib
```

This notebook assumes that `tickets.csv` is in the same folder as the notebook.


## Learning Goals

By the end of this lesson, you should be able to:

1. Distinguish between Markdown cells and code cells.
2. Explain notebook kernel state and cell execution order.
3. Use strings, lists, dictionaries, loops, and functions to handle simple data.
4. Use Pandas to read and inspect a CSV file.
5. Explain features, labels, training sets, test sets, and predictions.
6. Build a keyword baseline.
7. Run a TF-IDF + Logistic Regression text classification model.
8. Compare a baseline and a trained model using accuracy and error examples.
9. Explain how a trained model can be used inside an application pipeline.
10. Use SQLite to retrieve pending tickets and save prediction results.
11. Separate model prediction from validation and business rules.
12. Check AI-generated code for errors, assumptions, and data leakage.
13. Use Restart Kernel and Run All to verify that a notebook is reproducible.


## Classroom Map

This notebook is longer than the minimum code needed to train a model. That is intentional.

During class, use the main sections for the live walkthrough. Use the checkpoint cells for short pauses, partner checks, or written answers. Use the appendices when the class is ready for more mathematical detail.

| Section | Main question | What students should notice |
|---|---|---|
| Notebook mental model | Why can notebooks behave differently after a restart? | Variables live in kernel memory |
| Python basics | What Python objects represent a ticket? | Text, labels, dictionaries, lists, and functions |
| Dataset inspection | What data do we actually have? | Rows, columns, labels, missing values |
| Baseline | What can simple rules do? | Baselines are explainable but limited |
| Train/test split | How do we make evaluation fair? | Test data must stay unseen |
| Model training | How does text become a prediction? | TF-IDF creates features; Logistic Regression learns weights |
| Evaluation | Did the model improve? | Accuracy is useful but incomplete |
| Application pipeline | What happens after a model predicts? | Apps validate input, apply rules, and save state |
| AI review | Can we trust generated code? | Code must be runnable, explainable, and leakage-free |


## Key Vocabulary

Keep this table nearby. These terms will come back throughout the notebook.

| Term | Meaning in this project |
|---|---|
| example | one row in `tickets.csv` |
| feature | information the model can use, here words from the ticket text |
| label | the correct category: `account`, `billing`, `technical`, or `other` |
| baseline | a simple reference system that the trained model should beat |
| training set | examples used to fit the model |
| test set | examples held back for evaluation |
| prediction | the label returned by the baseline or model |
| leakage | using test information during training or model design |
| database | a place where application records can be stored and updated |
| status | a field that tracks whether a ticket is pending, classified, or failed |
| business rule | application logic that uses a model result to make a product decision |
| reproducible | able to restart the kernel and run every cell from the top with no hidden state |


## 0. Project Goal

By the end, we want to run:

```python
predict_ticket("My payment was charged twice.")
```

And get something like:

```python
"billing"
```

The final function should follow this simple contract:

| Function input | Function output |
|---|---|
| one string, such as `"The app will not open"` | one allowed label, such as `"technical"` |

Do not rush into model code. We will start with small Python pieces, then connect them into a pipeline step by step.


In [ ]:
print("Today we will build a ticket classifier step by step.")


### Minimum Success Criteria

By the end of the notebook, we should be able to say yes to all of these:

- The notebook can restart and run from top to bottom.
- The dataset is loaded from `tickets.csv`, not manually copied into the notebook.
- The data schema is checked before model training.
- A keyword baseline is built before the trained model.
- The model trains only on training data.
- Baseline and model are evaluated on the same test data.
- We inspect at least a few wrong predictions.
- The final `predict_ticket()` function handles invalid input.
- The application workflow fetches pending tickets, validates them, predicts labels, and saves results.


## 1. Notebook Mental Model

A notebook has three main visible parts:

| Part | Meaning |
|---|---|
| Markdown cell | Writes headings, explanations, questions, and conclusions |
| Code cell | Runs Python code |
| Output | Shows the result of running code |

The kernel is the Python process behind the notebook. It remembers variables that have already been created.


In [ ]:
course_name = "AI-Assisted Coding"
print(course_name)


The value above is now stored in kernel memory. We can check whether the name exists by looking at `globals()`, which is Python's dictionary of currently known names.

You do not need to memorize `globals()`. It is here to make the invisible kernel state visible.


In [ ]:
names_to_check = ["course_name", "ticket_label_before_created"]

for name in names_to_check:
    print(name, "exists in kernel memory?", name in globals())


### Execution Order Experiment

If we use a variable before creating it, Python raises a `NameError`.

The next cell uses `try` / `except` to show the error without stopping the entire notebook.


In [ ]:
try:
    print(ticket_label_before_created)
except NameError as error:
    print("Expected NameError:", error)


In [ ]:
ticket_label_before_created = "account"
print(ticket_label_before_created)


**Key idea**

A notebook does not automatically run in page order. It runs in the order you actually execute the cells.

If one cell runs successfully, that does not prove the whole notebook can run from a clean state.


### Checkpoint: Notebook State

Answer in one sentence:

```text
Why can a notebook work on the author's computer but fail after another person restarts it?
```

Useful words to include: `kernel`, `variable`, `execution order`, `file path`.


## 2. Python Basics in Context

This section covers only the Python concepts we need immediately for this project:

- string
- variable
- dictionary
- list
- loop
- function


### 2.1 String and Variables


In [ ]:
ticket_text = "I forgot my password"
ticket_label = "account"

print(ticket_text)
print(ticket_label)


Two string operations will be especially useful later:

- `.lower()` makes matching less sensitive to uppercase/lowercase differences.
- `"word" in text` checks whether a word or phrase appears inside a string.


In [ ]:
message = "I Forgot My Password"

print(message.lower())
print("password" in message.lower())
print("payment" in message.lower())


### 2.2 Dictionary

In this project, one ticket can be represented as a dictionary:

- `text` is the input message.
- `label` is the correct answer.


In [ ]:
ticket = {
    "text": "I forgot my password",
    "label": "account",
}

print(ticket["text"])
print(ticket["label"])


A dictionary has keys and values. In a dataset, column names play a similar role.

If AI-generated code invents a key or column name that does not exist, the code may fail or silently analyze the wrong thing.


In [ ]:
print("Dictionary keys:", list(ticket.keys()))
print("Model input:", ticket["text"])
print("Target label:", ticket["label"])


### 2.3 List and Loop


In [ ]:
tickets = [
    {"text": "I forgot my password", "label": "account"},
    {"text": "I was charged twice", "label": "billing"},
    {"text": "The app keeps crashing", "label": "technical"},
]

for ticket in tickets:
    print(ticket["text"])


A list lets us store many examples. A loop lets us apply the same action to every example.

Later, we will use the same idea to ask the baseline and model to predict many test examples.


In [ ]:
for index, ticket in enumerate(tickets):
    print(index, ticket["label"], "-", ticket["text"])


### 2.4 Function


In [ ]:
def show_ticket(ticket):
    print("Message:", ticket["text"])
    print("Correct label:", ticket["label"])


show_ticket(tickets[0])


Some functions mainly have an effect, such as printing. Other functions return a value.

A classifier function should return a prediction so another part of the pipeline can use it.


In [ ]:
def get_ticket_label(ticket):
    return ticket["label"]


first_label = get_ticket_label(tickets[0])
print(first_label)


### Quick Practice

Change the text and label in `practice_ticket`, then run the cell again.


In [ ]:
practice_ticket = {
    "text": "I cannot connect to campus wifi",
    "label": "technical",
}

show_ticket(practice_ticket)


## 3. Load the Dataset

Real projects do not usually work with only three dictionaries. We store many tickets in a CSV file.

`tickets.csv` must contain at least two columns:

- `text`
- `label`


In [ ]:
from pathlib import Path

import pandas as pd


def show(value):
    try:
        display(value)
    except NameError:
        print(value)

DATA_PATH = Path("tickets.csv")

assert DATA_PATH.exists(), f"Cannot find data file: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)
show(df.head())


A Pandas `DataFrame` is a table.

- Each row is one ticket example.
- Each column is one kind of information.
- `text` is the model input.
- `label` is the correct answer.


In [ ]:
first_row = df.iloc[0]

print("First row as a dictionary:")
print(first_row.to_dict())


### Inspect the Dataset


In [ ]:
print("Shape:", df.shape)
print("Columns:", list(df.columns))

print("\nLabel counts:")
show(df["label"].value_counts())

print("\nMissing values:")
show(df.isna().sum())


The label distribution matters. If one label appears much more often than the others, a model can get deceptively high accuracy by mostly guessing the common label.


In [ ]:
label_summary = pd.DataFrame({
    "count": df["label"].value_counts(),
    "percent": (df["label"].value_counts(normalize=True) * 100).round(1),
})

show(label_summary)


### Student Check

Answer these questions in a new Markdown cell:

1. How many examples are in the dataset?
2. What is the target column?
3. Are there any missing values?
4. Which label appears most often?


### Data Schema Contract

A schema describes what columns and values we expect. In this project, the minimum schema is:

| Column | Type | Meaning | Required? |
|---|---|---|---|
| `text` | string | the helpdesk request written by a student | yes |
| `label` | string | the correct ticket category | yes |

Allowed labels:

```text
account, billing, technical, other
```


### Validate the Dataset


In [ ]:
required_columns = {"text", "label"}
ALLOWED_LABELS = {"account", "billing", "technical", "other"}

assert required_columns.issubset(df.columns), "Dataset must contain text and label columns."
assert df["text"].notna().all(), "Text column cannot contain missing values."
assert df["label"].notna().all(), "Label column cannot contain missing values."

df = df.copy()

df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip().str.lower()

assert df["text"].ne("").all(), "Every row needs non-empty text."
assert df["label"].ne("").all(), "Every row needs a non-empty label."
assert set(df["label"]).issubset(ALLOWED_LABELS), "Found a label outside the allowed set."

print("Dataset validation passed.")


We can also check for duplicate examples. Duplicates are not always wrong, but they can make a small dataset look larger than it really is.


In [ ]:
duplicate_count = df.duplicated(subset=["text", "label"]).sum()

print("Duplicate text/label rows:", duplicate_count)
assert duplicate_count == 0, "This starter dataset should not contain duplicate text/label rows."


**Key idea**

A pipeline should not silently continue when the input data is wrong.


## 4. Build a Keyword Baseline

Before training a model, we will first write a very simple rule-based system.

The purpose of a baseline is not to be perfect. It gives us something to compare the trained model against.


In [ ]:
def keyword_baseline(text):
    text = text.lower()

    if (
        "password" in text
        or "login" in text
        or "reset" in text
        or "account" in text
        or "profile" in text
        or "sign in" in text
    ):
        return "account"
    elif (
        "payment" in text
        or "charged" in text
        or "charge" in text
        or "refund" in text
        or "billing" in text
        or "invoice" in text
        or "tuition" in text
        or "card" in text
        or "fee" in text
    ):
        return "billing"
    elif (
        "crash" in text
        or "error" in text
        or "frozen" in text
        or "connect" in text
        or "wifi" in text
        or "will not open" in text
        or "app" in text
        or "website" in text
        or "upload" in text
        or "server" in text
    ):
        return "technical"
    else:
        return "other"


In [ ]:
assert keyword_baseline("I forgot my password") == "account"
assert keyword_baseline("I was charged twice") == "billing"
assert keyword_baseline("The app keeps crashing") == "technical"
assert keyword_baseline("Where is the library?") == "other"

print("Baseline tests passed.")


The baseline makes decisions in order. This order matters because the first matching category wins.

| Order | Category | Example keywords |
|---|---|---|
| 1 | `account` | `password`, `login`, `reset`, `profile` |
| 2 | `billing` | `payment`, `charged`, `refund`, `invoice` |
| 3 | `technical` | `crash`, `error`, `wifi`, `server` |
| 4 | `other` | no keyword matched |

A useful habit is to ask not only "What did it predict?" but also "Why did it predict that?"


In [ ]:
BASELINE_RULES = [
    ("account", ["password", "login", "reset", "account", "profile", "sign in"]),
    ("billing", ["payment", "charged", "charge", "refund", "billing", "invoice", "tuition", "card", "fee"]),
    ("technical", ["crash", "error", "frozen", "connect", "wifi", "will not open", "app", "website", "upload", "server"]),
]


def explain_keyword_baseline(text):
    normalized_text = text.lower()

    for label, keywords in BASELINE_RULES:
        matched_keywords = [
            keyword
            for keyword in keywords
            if keyword in normalized_text
        ]

        if matched_keywords:
            return {
                "prediction": label,
                "matched_keywords": matched_keywords,
            }

    return {
        "prediction": "other",
        "matched_keywords": [],
    }


In [ ]:
example_messages = [
    "I cannot access my profile.",
    "Refund my payment please.",
    "The website is frozen.",
    "Can I bring a guest to campus?",
]

for message in example_messages:
    print(message, "->", keyword_baseline(message))


In [ ]:
explanation_rows = []

for message in example_messages:
    explanation = explain_keyword_baseline(message)
    explanation_rows.append({
        "text": message,
        "prediction": explanation["prediction"],
        "matched_keywords": ", ".join(explanation["matched_keywords"]) or "none",
    })

show(pd.DataFrame(explanation_rows))


### Baseline Limitation

Keyword rules can fail when:

- a message uses a word we did not include,
- the same word appears in multiple categories,
- the message is vague or short.


### Baseline Practice

Try these messages mentally before running any code:

1. `"I cannot enter the student portal"`
2. `"My refund is still pending"`
3. `"The quiz page freezes after I click submit"`

For each one, ask:

- Which keyword would match?
- Which label would the baseline return?
- Is the returned label reasonable?


## 5. Create a Train/Test Split

Machine learning uses examples to learn patterns.

| Name | Meaning |
|---|---|
| `X` | model input, here the ticket text |
| `y` | correct answer, here the ticket label |
| training set | data used to train the model |
| test set | data held back for evaluation |

**Rule:** Test data must remain unseen during training.

Two arguments deserve special attention:

- `random_state=42` makes the split reproducible.
- `stratify=y` keeps the label proportions similar in the training and test sets.


In [ ]:
X = df["text"]
y = df["label"]

print("First input:", X.iloc[0])
print("First label:", y.iloc[0])


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Training examples:", len(X_train))
print("Test examples:", len(X_test))

assert len(X_train) + len(X_test) == len(df)
assert set(X_train.index).isdisjoint(set(X_test.index))


After splitting, check that every label still appears in both training and test data.


In [ ]:
split_summary = pd.DataFrame({
    "train_count": y_train.value_counts().sort_index(),
    "test_count": y_test.value_counts().sort_index(),
})

split_summary["train_percent"] = (
    y_train.value_counts(normalize=True).sort_index() * 100
).round(1)

split_summary["test_percent"] = (
    y_test.value_counts(normalize=True).sort_index() * 100
).round(1)

show(split_summary)

assert set(y_train) == ALLOWED_LABELS
assert set(y_test) == ALLOWED_LABELS


### Data Leakage Check

In the situations below, the second one is clear data leakage:

| Situation | Problem? |
|---|---|
| Train only on `X_train` and `y_train` | OK |
| Train on all `X` and `y`, then evaluate on `X_test` | Data leakage |
| Repeatedly change the model after looking at test results | Test set contamination |

A common AI-generated mistake looks like this:

```python
model.fit(X, y)          # wrong: trains on all examples
model.predict(X_test)    # test examples were already seen
```

The correct pattern is:

```python
model.fit(X_train, y_train)
model.predict(X_test)
```


## 6. Train a Text Classification Model

Model pipeline:

```text
Ticket text -> TF-IDF vectorizer -> Numbers -> Logistic regression -> Predicted label
```

We will not go deep into the math yet. First, focus on what each step does inside the pipeline.


Before training the real model, look at a tiny vectorizer example. The output is a table of numbers, one column per word.

This is the bridge between text and machine learning: text must become numeric features before the classifier can learn.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

demo_texts = [
    "password reset",
    "payment refund",
    "app error",
]

demo_vectorizer = TfidfVectorizer()
demo_matrix = demo_vectorizer.fit_transform(demo_texts)

demo_features = pd.DataFrame(
    demo_matrix.toarray(),
    columns=demo_vectorizer.get_feature_names_out(),
    index=demo_texts,
).round(3)

show(demo_features)


Now build the real model pipeline.

Keeping `TfidfVectorizer` inside `Pipeline` is important. When we call `model.fit(X_train, y_train)`, the vectorizer learns its vocabulary only from training text, not from test text.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("vectorizer", TfidfVectorizer()),
    ("classifier", LogisticRegression(max_iter=1000)),
])

model


In [ ]:
model.fit(X_train, y_train)

model_predictions = model.predict(X_test)

assert len(model_predictions) == len(X_test)
assert set(model_predictions).issubset(set(y))

print("Model training and prediction complete.")


After fitting, the pipeline contains learned pieces:

- the vectorizer has a vocabulary of words from the training text,
- the classifier has learned weights that connect word features to labels.


In [ ]:
trained_vectorizer = model.named_steps["vectorizer"]
trained_classifier = model.named_steps["classifier"]

print("Model classes:", list(trained_classifier.classes_))
print("Vocabulary size:", len(trained_vectorizer.get_feature_names_out()))
print("First 20 vocabulary terms:")
print(list(trained_vectorizer.get_feature_names_out()[:20]))


In [ ]:
import numpy as np

feature_names = np.array(trained_vectorizer.get_feature_names_out())
top_feature_rows = []

for class_index, class_label in enumerate(trained_classifier.classes_):
    class_weights = trained_classifier.coef_[class_index]
    top_indices = class_weights.argsort()[-8:][::-1]

    for feature_index in top_indices:
        top_feature_rows.append({
            "label": class_label,
            "feature": feature_names[feature_index],
            "weight": round(class_weights[feature_index], 3),
        })

top_features = pd.DataFrame(top_feature_rows)
show(top_features)


The table above should not be treated as a perfect explanation, but it helps students see that the model learned associations such as payment-related words for `billing` or app/network words for `technical`.


In [ ]:
probability_examples = [
    "My payment was charged twice.",
    "The campus app will not open.",
    "How do I reserve a study room?",
]

probability_table = pd.DataFrame(
    model.predict_proba(probability_examples),
    columns=model.classes_,
    index=probability_examples,
).round(3)

show(probability_table)


**Key idea**

- `fit()` means learn from training examples.
- `predict()` means use the learned pattern on new inputs.
- `predict_proba()` shows the model's estimated probabilities for each label.


## 7. Compare Baseline and Model


In [ ]:
from sklearn.metrics import accuracy_score

baseline_predictions = [
    keyword_baseline(text)
    for text in X_test
]

baseline_accuracy = accuracy_score(y_test, baseline_predictions)
model_accuracy = accuracy_score(y_test, model_predictions)

print("Baseline accuracy:", round(baseline_accuracy, 3))
print("Model accuracy:", round(model_accuracy, 3))


Accuracy means:

```text
number of correct predictions / number of total predictions
```

Accuracy is easy to understand, but it hides which categories are being confused.


In [ ]:
comparison_table = pd.DataFrame({
    "system": ["keyword baseline", "trained model"],
    "accuracy": [baseline_accuracy, model_accuracy],
    "correct_predictions": [
        int((y_test == baseline_predictions).sum()),
        int((y_test == model_predictions).sum()),
    ],
    "total_test_examples": [len(y_test), len(y_test)],
})

comparison_table["accuracy"] = comparison_table["accuracy"].round(3)
show(comparison_table)


Both systems must use the same test set. Otherwise, the comparison is not fair.


In [ ]:
from sklearn.metrics import confusion_matrix

labels = sorted(ALLOWED_LABELS)
matrix = confusion_matrix(
    y_test,
    model_predictions,
    labels=labels,
)

confusion_table = pd.DataFrame(
    matrix,
    index=[f"actual_{label}" for label in labels],
    columns=[f"predicted_{label}" for label in labels],
)

show(confusion_table)

# Optional plot version for a Jupyter environment with Matplotlib:
#
# import matplotlib.pyplot as plt
# from sklearn.metrics import ConfusionMatrixDisplay
#
# ConfusionMatrixDisplay.from_predictions(
#     y_test,
#     model_predictions,
#     labels=labels,
#     xticks_rotation=45,
# )
# plt.title("Model Confusion Matrix")
# plt.tight_layout()
# plt.show()


### Inspect Prediction Errors


In [ ]:
results = pd.DataFrame({
    "text": X_test,
    "correct_label": y_test,
    "baseline_prediction": baseline_predictions,
    "model_prediction": model_predictions,
}).reset_index(drop=True)

results["baseline_correct"] = (
    results["correct_label"] == results["baseline_prediction"]
)

results["model_correct"] = (
    results["correct_label"] == results["model_prediction"]
)

show(results.head())


In [ ]:
model_errors = results[results["model_correct"] == False]
baseline_errors = results[results["baseline_correct"] == False]

print("Baseline errors:", len(baseline_errors))
print("Model errors:", len(model_errors))

if len(model_errors) > 0:
    show(model_errors.head(10))
else:
    print("No model errors in this split. Use baseline errors for error-analysis practice.")
    show(baseline_errors.head(10))


A deeper comparison asks two questions:

- Which baseline mistakes did the trained model fix?
- Which examples did the trained model miss, even if the baseline got them right?


In [ ]:
fixed_by_model = results[
    (results["baseline_correct"] == False)
    & (results["model_correct"] == True)
]

missed_by_model = results[
    (results["baseline_correct"] == True)
    & (results["model_correct"] == False)
]

print("Baseline wrong, model correct:", len(fixed_by_model))
show(fixed_by_model.head(10))

print("\nBaseline correct, model wrong:", len(missed_by_model))
show(missed_by_model.head(10))


### Student Error Analysis

Choose one error example and answer these questions in a Markdown cell:

1. What did the system predict?
2. What was the correct label?
3. Why might the system have made this mistake?
4. Would adding more data help? Why or why not?


## 8. Model Interface: `predict_ticket()`

The trained model is useful, but application code should not call raw model logic everywhere.

We wrap the model in a small function that checks the input and checks the model output.

This section is the bridge between the training pipeline and the application pipeline.


In [ ]:
def predict_ticket(text):
    if not isinstance(text, str):
        raise TypeError("text must be a string")

    if not text.strip():
        raise ValueError("text cannot be empty")

    prediction = model.predict([text])[0]

    if prediction not in ALLOWED_LABELS:
        raise ValueError("model returned an invalid label")

    return prediction


In [ ]:
print(predict_ticket("My payment was charged twice."))
print(predict_ticket("The campus app will not open."))

assert predict_ticket("I forgot my password") in ALLOWED_LABELS
assert predict_ticket("The app will not open") in ALLOWED_LABELS

print("predict_ticket tests passed.")


The function should also fail clearly for invalid input. Clear failure is better than a confusing model error several lines later.


In [ ]:
invalid_inputs = [
    "",
    "   ",
    None,
    123,
]

for invalid_input in invalid_inputs:
    try:
        predict_ticket(invalid_input)
    except (TypeError, ValueError) as error:
        print(repr(invalid_input), "->", type(error).__name__, "-", error)
    else:
        raise AssertionError(f"Expected an error for {invalid_input!r}")


## 9. From a Trained Model to an Application

The training pipeline produces a model:

```text
CSV -> validation -> train/test split -> fit -> evaluate -> model
```

The application pipeline uses the model:

```text
database -> retrieve ticket -> validate -> predict -> apply business rule -> save result
```

Discussion questions:

1. Does every new ticket require retraining the model?
2. Does the application need `y_train`?
3. Should predictions only print to the screen, or should they be saved?
4. Which errors should be caught before calling the model?


**Important distinction**

Training code answers:

```text
How do we create a model?
```

Application code answers:

```text
How do we use the model safely when new records arrive?
```


## 10. Create a Simple Application Database

In the first half of the notebook, `tickets.csv` represented historical training data.

In an application, new tickets keep arriving and their status changes over time. A database is a better fit for that kind of state.

We will use SQLite because it is built into Python. No database server is required.


In [ ]:
import sqlite3

DB_PATH = Path("tickets_app.db")

connection = sqlite3.connect(DB_PATH)

connection.execute("DROP TABLE IF EXISTS tickets")

connection.execute(
    """
    CREATE TABLE tickets (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        text TEXT NOT NULL,
        predicted_label TEXT,
        needs_review INTEGER,
        status TEXT NOT NULL DEFAULT 'pending',
        error_message TEXT
    )
    """
)

sample_tickets = [
    ("I cannot reset my password.",),
    ("My card was charged twice.",),
    ("The application freezes when I open it.",),
    ("Where can I find the campus library?",),
    ("   ",),
]

connection.executemany(
    """
    INSERT INTO tickets (text)
    VALUES (?)
    """,
    sample_tickets,
)

connection.commit()

print(f"Created application database at: {DB_PATH}")


In [ ]:
show(
    pd.read_sql_query(
        """
        SELECT
            id,
            text,
            predicted_label,
            needs_review,
            status,
            error_message
        FROM tickets
        ORDER BY id
        """,
        connection,
    )
)


The table has two different kinds of fields:

| Field | Purpose |
|---|---|
| `id` | stable identifier for one ticket |
| `text` | the message submitted by the user |
| `predicted_label` | the model output, saved later |
| `needs_review` | application decision, saved later |
| `status` | current processing state |
| `error_message` | failure details if processing fails |

CSV is mainly for training examples. A database is for application records that change over time.


## 11. Fetch Pending Tickets

The application should only process tickets that are still waiting for work.

The next function turns database rows into Python dictionaries:

```text
database row -> Python dictionary -> application object
```


In [ ]:
def fetch_pending_tickets(connection, limit=10):
    if not isinstance(limit, int):
        raise TypeError("limit must be an integer")

    if limit <= 0:
        raise ValueError("limit must be greater than zero")

    rows = connection.execute(
        """
        SELECT id, text
        FROM tickets
        WHERE status = ?
        ORDER BY id
        LIMIT ?
        """,
        ("pending", limit),
    ).fetchall()

    return [
        {
            "id": row[0],
            "text": row[1],
        }
        for row in rows
    ]


In [ ]:
pending_tickets = fetch_pending_tickets(
    connection,
    limit=5,
)

show(pd.DataFrame(pending_tickets))


The query uses:

```sql
WHERE status = ?
```

That condition matters. The application pipeline should process pending records only, not records that have already been classified or failed.


## 12. Validation and Service Layer

Before calling the model, application code should check that the ticket object is valid.

Invalid data should fail early with a clear message.


In [ ]:
def validate_ticket(ticket):
    if not isinstance(ticket, dict):
        raise TypeError("ticket must be a dictionary")

    if "id" not in ticket:
        raise ValueError("ticket is missing id")

    if "text" not in ticket:
        raise ValueError("ticket is missing text")

    if not isinstance(ticket["id"], int):
        raise TypeError("ticket id must be an integer")

    if not isinstance(ticket["text"], str):
        raise TypeError("ticket text must be a string")

    if not ticket["text"].strip():
        raise ValueError("ticket text cannot be empty")


In [ ]:
validate_ticket({
    "id": 1,
    "text": "I cannot login",
})

try:
    validate_ticket({
        "id": 2,
        "text": "",
    })
except ValueError as error:
    print("Expected validation error:", error)
else:
    raise AssertionError("Empty ticket text should raise an error")


Now we wrap the model call in an application function.

This function has three responsibilities:

1. Validate the incoming ticket.
2. Ask the trained model for a prediction.
3. Apply a small business rule.

The business rule is:

```python
needs_review = predicted_label == "other"
```

This is not a model training rule. It is application logic.


In [ ]:
def classify_ticket(ticket, trained_model):
    validate_ticket(ticket)

    predicted_label = trained_model.predict(
        [ticket["text"]]
    )[0]

    predicted_label = str(predicted_label)

    if predicted_label not in ALLOWED_LABELS:
        raise ValueError(
            f"Invalid model output: {predicted_label}"
        )

    needs_review = predicted_label == "other"

    return {
        "ticket_id": ticket["id"],
        "predicted_label": predicted_label,
        "needs_review": needs_review,
        "status": "classified",
    }


In [ ]:
demo_result = classify_ticket(
    {
        "id": 100,
        "text": "I forgot my password",
    },
    model,
)

demo_result


| Component | Responsibility |
|---|---|
| `validate_ticket()` | check whether the input is usable |
| `model.predict()` | produce a label from text |
| `classify_ticket()` | connect the model to the application workflow |
| business rule | decide whether a human should review the ticket |


## 13. Save Classification Results

A prediction is only useful to the application if we store it somewhere.

The next function updates the database record for one pending ticket.


In [ ]:
def save_classification(connection, result):
    with connection:
        cursor = connection.execute(
            """
            UPDATE tickets
            SET predicted_label = ?,
                needs_review = ?,
                status = ?,
                error_message = NULL
            WHERE id = ?
              AND status = 'pending'
            """,
            (
                result["predicted_label"],
                int(result["needs_review"]),
                result["status"],
                result["ticket_id"],
            ),
        )

    if cursor.rowcount != 1:
        raise RuntimeError(
            "Ticket was not updated. "
            "It may already have been processed."
        )


def mark_ticket_failed(connection, ticket_id, error_message):
    with connection:
        connection.execute(
            """
            UPDATE tickets
            SET status = ?,
                error_message = ?
            WHERE id = ?
            """,
            (
                "failed",
                str(error_message),
                ticket_id,
            ),
        )


The `UPDATE` query includes:

```sql
WHERE id = ?
  AND status = 'pending'
```

That prevents the function from silently re-processing a ticket that has already moved out of the pending state.


## 14. Run the Application Pipeline

Now we can connect the application steps:

```text
fetch pending tickets -> validate -> classify -> save result
```

If one ticket is invalid, the whole batch should not necessarily stop. We can mark that one ticket as failed and continue.


In [ ]:
def process_pending_tickets(
    connection,
    trained_model,
    limit=10,
):
    tickets = fetch_pending_tickets(
        connection,
        limit=limit,
    )

    results = []

    for ticket in tickets:
        try:
            result = classify_ticket(
                ticket,
                trained_model,
            )

            save_classification(
                connection,
                result,
            )

            results.append(result)

        except (TypeError, ValueError) as error:
            mark_ticket_failed(
                connection,
                ticket["id"],
                error,
            )

            results.append({
                "ticket_id": ticket["id"],
                "status": "failed",
                "error": str(error),
            })

    return results


In [ ]:
application_results = process_pending_tickets(
    connection,
    model,
    limit=10,
)

show(pd.DataFrame(application_results))


In [ ]:
application_state = pd.read_sql_query(
    """
    SELECT
        id,
        text,
        predicted_label,
        needs_review,
        status,
        error_message
    FROM tickets
    ORDER BY id
    """,
    connection,
)

show(application_state)


Expected state changes:

| Ticket type | State change |
|---|---|
| valid ticket | `pending` -> `classified` |
| blank ticket text | `pending` -> `failed` |

This is the point where the notebook starts to look like software engineering rather than only model training.


## 15. Application Assertions

We do not need `pytest` yet. Notebook assertions are enough to check the most important behavior.


In [ ]:
test_ticket = {
    "id": 999,
    "text": "I was charged twice",
}

test_result = classify_ticket(
    test_ticket,
    model,
)

assert test_result["ticket_id"] == 999
assert test_result["predicted_label"] in ALLOWED_LABELS
assert isinstance(test_result["needs_review"], bool)
assert test_result["status"] == "classified"

try:
    validate_ticket({
        "id": 999,
        "text": "",
    })

    assert False, "Empty ticket text should raise an error"

except ValueError:
    pass

remaining_pending = connection.execute(
    """
    SELECT COUNT(*)
    FROM tickets
    WHERE status = 'pending'
    """
).fetchone()[0]

classified_count = connection.execute(
    """
    SELECT COUNT(*)
    FROM tickets
    WHERE status = 'classified'
    """
).fetchone()[0]

failed_count = connection.execute(
    """
    SELECT COUNT(*)
    FROM tickets
    WHERE status = 'failed'
    """
).fetchone()[0]

assert remaining_pending == 0
assert classified_count == 4
assert failed_count == 1

print("Application assertions passed.")


### Failure Case Activity

Assign one case to each group:

| Group | Failure case |
|---|---|
| A | ticket text is empty |
| B | ticket dictionary is missing `text` |
| C | model returns an unknown label |
| D | ticket has already been classified |
| E | database query uses the wrong column name |

Each group answers:

1. Which function should detect the problem?
2. Should the whole program stop?
3. What status should be stored?
4. What information should the caller receive?


## 16. AI Workflow Comparison

In the final part of class, students can work in groups. Each group reviews one AI-generated workflow.

### Workflow A: One-shot generation

```text
Write a complete Python notebook that trains
a text classifier for the tickets.csv dataset.
```

### Workflow B: Plan first

```text
First inspect the task and write a detailed plan.
Identify the expected dataset columns and evaluation steps.
Then implement the notebook one section at a time.
```

### Workflow C: Skill-guided generation

```text
Before writing code:

1. State the expected dataset schema.
2. Do not invent column names.
3. Build a baseline first.
4. Use a train/test split with random_state=42.
5. Add one validation check after every stage.
6. Do not train on the test data.
7. Finish with Restart and Run All instructions.
```


### AI Code Review Checklist

| Check item | Yes/No | Evidence |
|---|---|---|
| Can Restart Kernel and Run All | | |
| Uses real column names | | |
| Builds a baseline | | |
| Correctly separates train and test data | | |
| No obvious data leakage | | |
| Includes assertions | | |
| Students can explain the main code | | |
| Errors can be fixed locally | | |

One-sentence report:

```text
The most serious problem we found was ______.
```


### Common AI-Generated Notebook Failure Modes

| Failure mode | Why it matters | How to check |
|---|---|---|
| Invented column names | Code may not match the real dataset | Compare code to `df.columns` |
| No baseline | Model has no reference point | Search for a rule-based or simple comparison system |
| Train/test leakage | Evaluation becomes too optimistic | Check whether `fit()` uses only training data |
| Hidden state | Notebook works only after out-of-order execution | Restart Kernel and Run All |
| No assertions | Wrong results may pass silently | Look for checks after loading, splitting, predicting |
| Overly complex code | Beginners cannot explain or fix it | Ask each student to explain one cell |
| Unclear file paths | Code may only work on one computer | Check whether files are local and relative |


### Group Review Prompt

When reviewing an AI-generated workflow, do not start by asking "Is the code impressive?"

Start with these questions:

1. What dataset schema does the code assume?
2. Where does the train/test split happen?
3. Which exact line trains the model?
4. Which exact line evaluates the model?
5. What evidence shows the notebook is reproducible?


## 17. Restart and Run All Checklist

Before submitting or sharing:

- Restart the kernel.
- Run all cells from top to bottom.
- Confirm no errors.
- Confirm `tickets.csv` is in the same folder as this notebook.
- Confirm the final `predict_ticket()` examples return allowed labels.
- Confirm the SQLite application table has no pending tickets after processing.
- In `PROMPTS.md`, record what AI helped with and what you changed.


## Appendix A: Manual TF-IDF Calculation

This section is optional supplement material.

Main idea:

- **TF** asks: how often does a word appear in this document?
- **IDF** asks: how rare is this word across all documents?
- **TF-IDF** is higher when a word is frequent in one document but not common everywhere.
- A word that appears in almost every document is usually less useful for classification.

Scikit-learn's smoothed IDF formula is:

$$
\text{idf}(t) = \log\left(\frac{1 + n}{1 + \text{df}(t)}\right) + 1
$$

where:

- `n` is the number of documents.
- `df(t)` is the number of documents containing term `t`.

We will calculate a tiny example by hand, then compare it to `TfidfVectorizer`.


In [ ]:
import math

mini_corpus = [
    "password reset password",
    "payment refund",
    "password login",
]


def tokenize(text):
    return text.lower().split()


mini_tokens = [tokenize(doc) for doc in mini_corpus]
vocabulary = sorted({token for tokens in mini_tokens for token in tokens})
n_docs = len(mini_corpus)

token_count_rows = []
for tokens in mini_tokens:
    token_count_rows.append({
        term: tokens.count(term)
        for term in vocabulary
    })

token_counts = pd.DataFrame(
    token_count_rows,
    index=[
        "doc_0: password reset password",
        "doc_1: payment refund",
        "doc_2: password login",
    ],
)

print("Step 1: token counts")
show(token_counts)

doc_frequency = {}
for term in vocabulary:
    doc_frequency[term] = sum(term in tokens for tokens in mini_tokens)

idf = {
    term: math.log((1 + n_docs) / (1 + doc_frequency[term])) + 1
    for term in vocabulary
}

term_stats = pd.DataFrame({
    "document_frequency": doc_frequency,
    "idf": idf,
}).round(3)

print("Step 2: document frequency and IDF")
show(term_stats)

rows = []
for tokens in mini_tokens:
    row = {}
    for term in vocabulary:
        tf = tokens.count(term)
        row[term] = round(tf * idf[term], 3)
    rows.append(row)

manual_tfidf = pd.DataFrame(rows, index=[
    "doc_0: password reset password",
    "doc_1: payment refund",
    "doc_2: password login",
])

print("Step 3: raw TF-IDF = token count * IDF")
show(manual_tfidf)


In [ ]:
mini_vectorizer = TfidfVectorizer(
    norm=None,
    smooth_idf=True,
    token_pattern=r"(?u)\b\w+\b",
)

mini_matrix = mini_vectorizer.fit_transform(mini_corpus)

sklearn_tfidf = pd.DataFrame(
    mini_matrix.toarray(),
    columns=mini_vectorizer.get_feature_names_out(),
    index=manual_tfidf.index,
).round(3)

show(sklearn_tfidf)

assert list(manual_tfidf.columns) == list(sklearn_tfidf.columns)
assert manual_tfidf.equals(sklearn_tfidf)


The comparison above used `norm=None` so the numbers match our manual calculation.

By default, `TfidfVectorizer()` also normalizes each document vector. Normalization rescales each row so long documents do not automatically look more important just because they contain more words.


In [ ]:
import numpy as np

default_vectorizer = TfidfVectorizer(
    smooth_idf=True,
    token_pattern=r"(?u)\b\w+\b",
)

default_matrix = default_vectorizer.fit_transform(mini_corpus)

default_tfidf = pd.DataFrame(
    default_matrix.toarray(),
    columns=default_vectorizer.get_feature_names_out(),
    index=manual_tfidf.index,
).round(3)

show(default_tfidf)

row_lengths = [
    float(length)
    for length in np.sqrt((default_matrix.toarray() ** 2).sum(axis=1)).round(3)
]
print("L2 row lengths after default normalization:", list(row_lengths))


## Appendix B: Logistic Regression Math Derivation

This section is optional supplement material. It connects the model code to the math without requiring students to derive everything during the main class.

Logistic Regression has "regression" in the name, but in this notebook we use it for classification. The model calculates scores, turns those scores into probabilities, and chooses a label.

### Binary Case

Suppose each ticket has numeric features:

$$
x = [x_1, x_2, ..., x_m]
$$

Logistic regression first computes a score:

$$
z = w_1x_1 + w_2x_2 + ... + w_mx_m + b
$$

Then it converts the score into a probability with the sigmoid function:

$$
p = \sigma(z) = \frac{1}{1 + e^{-z}}
$$

If `p` is close to 1, the model predicts the positive class. If `p` is close to 0, it predicts the negative class.

### Loss Function

For one training example with correct answer `y`, where `y` is either 0 or 1:

$$
L = -\left[y\log(p) + (1-y)\log(1-p)\right]
$$

This loss is small when the model assigns high probability to the correct answer.

The model learns by adjusting `w` and `b` to reduce average loss across the training set.

The gradient for each weight has this useful shape:

$$
\frac{\partial L}{\partial w_j} = (p-y)x_j
$$

Interpretation:

- If prediction `p` is too high, reduce weights connected to active features.
- If prediction `p` is too low, increase weights connected to active features.

### Multi-class Case

Our project has four labels, so scikit-learn uses a multi-class version.

Each label gets its own score:

$$
s_k = w_k \cdot x + b_k
$$

The scores are converted into probabilities with softmax:

$$
P(y=k|x) = \frac{e^{s_k}}{\sum_j e^{s_j}}
$$

The predicted label is the class with the highest probability.


In [ ]:
import numpy as np


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


toy_examples = pd.DataFrame({
    "contains_password": [1, 0, 1, 0],
    "contains_payment": [0, 1, 0, 1],
})

weights = np.array([1.4, -1.2])
bias = -0.2

scores = toy_examples.to_numpy() @ weights + bias
probabilities = sigmoid(scores)

toy_examples["score"] = scores.round(3)
toy_examples["probability_of_account"] = probabilities.round(3)

show(toy_examples)


The loss is larger when the model gives low probability to the correct answer.

In the next cell, `true_account` is the correct binary label: `1` means account, `0` means not account.


In [ ]:
toy_examples["true_account"] = [1, 0, 1, 0]

p = probabilities
y_binary = toy_examples["true_account"].to_numpy()

losses = -(
    y_binary * np.log(p)
    + (1 - y_binary) * np.log(1 - p)
)

toy_examples["loss"] = losses.round(3)

show(toy_examples)
print("Average loss:", round(losses.mean(), 3))


The gradient shape `(p - y) * x_j` gives an update direction.

If `p - y` is positive, the model was too confident in the positive class. If it is negative, the model was not confident enough.


In [ ]:
feature_matrix = toy_examples[["contains_password", "contains_payment"]].to_numpy()
errors = p - y_binary

average_gradient = (errors.reshape(-1, 1) * feature_matrix).mean(axis=0)

gradient_table = pd.DataFrame({
    "feature": ["contains_password", "contains_payment"],
    "average_gradient": average_gradient.round(3),
})

show(gradient_table)


For our four ticket labels, the model uses a multi-class version. Softmax turns several class scores into probabilities that add up to 1.


In [ ]:
def softmax(scores):
    shifted_scores = scores - np.max(scores)
    exp_scores = np.exp(shifted_scores)
    return exp_scores / exp_scores.sum()


class_scores = pd.Series({
    "account": 0.3,
    "billing": 2.1,
    "technical": 0.7,
    "other": -0.4,
})

class_probabilities = pd.DataFrame({
    "score": class_scores,
    "softmax_probability": softmax(class_scores.to_numpy()),
}).round(3)

show(class_probabilities)
print("Predicted label:", class_probabilities["softmax_probability"].idxmax())
print("Probability sum:", round(class_probabilities["softmax_probability"].sum(), 3))


### Connection Back to Our Pipeline

In the real model:

- `TfidfVectorizer` creates the numeric feature vector `x`.
- `LogisticRegression` learns weights for words and labels.
- `predict()` chooses the label with the strongest learned evidence.

We do not manually calculate these weights in normal code. We inspect, test, and validate the pipeline instead.


## Exit Ticket

Answer in 2-4 sentences each:

1. What is the difference between a baseline and a trained model?
2. Why should test data not be used to train the model?
3. When reviewing an AI-generated notebook, what would you check first?
